# Study Validation Pipeline
## Optimal Talking — Robustness & Design Selection

**Repo:** `github.com/catchshashank/optimal-nego`

This notebook runs the full validation pipeline (Sections 1–9) including a
**Stage 1B upgrade**: replacing the seed-lexicon emotion features with
`SamLowe/roberta-base-go_emotions` neural classifier probabilities.

### Sections
| # | Section | What it tests |
|---|---|---|
| 1 | Data & role validation | Role assignment, sale labels, short calls |
| 2 | Preprocessing variants | Backchannel filter configs |
| 1B | GoEmotions encoding | Neural emotion features (GPU) |
| 3 | Emotion measure comparison | Lexicon vs GoEmotions vs VADER vs LIWC |
| 4 | SSM parameter grid | k=2–6, window=1,3,5 |
| 5 | Annotation validation | Bargaining act coverage & outcome deltas |
| 6 | Outcome prediction grid | 12 nested models, full metrics |
| 7 | Robustness checks | Duration tertiles, controls, outliers |
| 8–9 | Decision rule | RETAIN / EXPLORATORY / DISCARD table |

> **Runtime:** ~25 min on Colab T4. Section 1B (GoEmotions) requires GPU.
> Section 4 (SSM grid) is the most compute-intensive (~15 min).

In [ ]:
# ── Setup: clone repo and install dependencies ─────────────────────────────
!git clone https://github.com/catchshashank/optimal-nego.git 2>/dev/null || \
  (cd /content/optimal-nego && git pull)

!pip install -q transformers torch tqdm scikit-learn pandas numpy scipy

import os
os.makedirs('/content/optimal-nego/outputs/validation', exist_ok=True)

DATA_PATH  = '/content/optimal-nego/data/nego-data-final.csv'
OUT_DIR    = '/content/optimal-nego/outputs/'
VAL_DIR    = '/content/optimal-nego/outputs/validation/'

print('Setup complete.')

In [ ]:
# ── Shared imports and constants ───────────────────────────────────────────
import numpy as np
import pandas as pd
import re, json, warnings
from itertools import product
from scipy import linalg
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score,
                              recall_score, f1_score, brier_score_loss)
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
warnings.filterwarnings('ignore')
np.random.seed(42)

# ── GoEmotions 28 labels (model output order) ──────────────────────────────
GOEMO_28 = [
    'admiration','amusement','anger','annoyance','approval','caring',
    'confusion','curiosity','desire','disappointment','disapproval','disgust',
    'embarrassment','excitement','fear','gratitude','grief','joy','love',
    'nervousness','optimism','pride','realization','relief','remorse',
    'sadness','surprise','neutral'
]

# 23 dims retained by SH-CCA in Stage 1 (drop admiration, relief, remorse,
# surprise, neutral — not reliably recovered from this corpus)
DROP_DIMS  = {'admiration','relief','remorse','surprise','neutral'}
DIMS_23    = [d for d in GOEMO_28 if d not in DROP_DIMS]

# Seed-lexicon patterns (Stage 1A — fallback when GPU unavailable)
EMOTION_SEEDS = {
    'amusement':      ['funny','laugh','joke','amusing','humor','witty'],
    'anger':          ['unacceptable','ridiculous','furious','angry','frustrated','unfair'],
    'annoyance':      ['come on','seriously','stop','enough','listen','fine'],
    'approval':       ['agree','okay','sure','absolutely','correct','right','fair','deal'],
    'caring':         ['help','support','understand','concern','assist','important'],
    'confusion':      ['confused','not sure','unclear','clarify','explain','follow'],
    'curiosity':      ['wonder','curious','how','why','interested','question'],
    'desire':         ['want','wish','hope','need','prefer','would like','goal'],
    'disappointment': ['disappointed','unfortunately','expected','hoped','shame','missed'],
    'disapproval':    ['disagree','reject','refuse','against','oppose','not okay'],
    'disgust':        ['awful','terrible','horrible','appalling','worst'],
    'embarrassment':  ['sorry','mistake','apologies','wrong','regret','excuse'],
    'excitement':     ['excited','wonderful','fantastic','amazing','thrilled','excellent'],
    'fear':           ['worried','afraid','concern','risk','nervous','uncertain'],
    'gratitude':      ['thank','grateful','appreciate','thankful','thanks'],
    'grief':          ['sad','loss','miss','unfortunate','difficult','painful'],
    'joy':            ['happy','pleased','delighted','glad','enjoy','satisfied'],
    'love':           ['love','adore','dear','devoted','cherish','fond'],
    'nervousness':    ['nervous','anxious','uneasy','tense','stress','worry'],
    'optimism':       ['hope','confident','positive','better','improve','believe'],
    'pride':          ['deserve','earned','value','maintain','position','worth'],
    'realization':    ['realize','makes sense','understand','get it','noted','recognize'],
    'sadness':        ['sad','unfortunate','shame','pity','regret','down'],
}
EMOTION_PATTERNS = {
    em: re.compile(r'\b(' + '|'.join(re.escape(s) for s in seeds) + r')\b', re.IGNORECASE)
    for em, seeds in EMOTION_SEEDS.items()
}

# Preprocessing
BC_RE_STRICT = re.compile(
    r'^\s*(yeah|mhm|mm|um|uh|okay|ok|yes|no|hi|hello|so|right|'
    r'sure|huh|ah|oh|hmm|yep|nope|alright|well|hey)\s*$', re.IGNORECASE)
PRICE_RE = re.compile(r'\$?\d[\d,\.]+|\bprice\b|\boffer\b|\bbid\b', re.IGNORECASE)
NUM_RE   = re.compile(r'\b(\d{3})\b')
NON_PRICE = {'213','233','239','240','235','225','150','200','300',
              '000','1846','1715','1875','1920'}

print(f'Retained emotion dims: {len(DIMS_23)}')
print(DIMS_23)

In [ ]:
# ── Shared utility functions ───────────────────────────────────────────────

def assign_roles(df):
    role_map = {}
    for cid, grp in df.groupby('conversation_id'):
        buyer = None
        for _, row in grp.sort_values('start_time').iterrows():
            if PRICE_RE.search(str(row['text'])):
                buyer = row['speaker_id']; break
        if buyer is None: buyer = 0
        spks   = grp['speaker_id'].unique()
        seller = [s for s in spks if s != buyer]
        seller = seller[0] if seller else (1 - buyer)
        role_map[cid] = {buyer: 'buyer', seller: 'seller'}
    df = df.copy()
    df['role'] = df.apply(
        lambda r: role_map.get(r['conversation_id'], {}).get(r['speaker_id'], 'buyer'), axis=1)
    return df, role_map

def extract_prices(text):
    prices = set()
    for m in NUM_RE.finditer(str(text)):
        if m.group(1) in NON_PRICE: continue
        v = int(m.group(1))
        if 150 <= v <= 300: prices.add(v)
    return prices

def lexicon_features(df, window=3, dims=None):
    """Seed-lexicon emotion features (Stage 1A)."""
    if dims is None: dims = list(EMOTION_PATTERNS.keys())
    df = df.copy()
    for em in dims:
        pat = EMOTION_PATTERNS[em]
        df[em] = df['text'].apply(lambda t: 1.0 if pat.search(str(t)) else 0.0)
    df = df.sort_values(['conversation_id','start_time']).reset_index(drop=True)
    for em in dims:
        df[em] = df.groupby('conversation_id')[em].transform(
            lambda x: x.rolling(window, min_periods=1, center=True).mean())
    return df

def evaluate_classifier(F, y, n_splits=5):
    """5-fold CV: AUC, accuracy, F1, Brier, 95% CI."""
    clf  = LogisticRegression(max_iter=2000, class_weight='balanced', C=0.1)
    F_sc = StandardScaler().fit_transform(np.nan_to_num(F))
    cv   = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    aucs, f1s, briers = [], [], []
    preds = np.zeros(len(y))
    for tr, te in cv.split(F_sc, y):
        clf.fit(F_sc[tr], y[tr])
        prob = clf.predict_proba(F_sc[te])[:,1]
        preds[te] = prob
        aucs.append(roc_auc_score(y[te], prob))
        f1s.append(f1_score(y[te], (prob>=0.5).astype(int), zero_division=0))
        briers.append(brier_score_loss(y[te], prob))
    return {
        'mean_auc':   round(np.mean(aucs),4),
        'std_auc':    round(np.std(aucs),4),
        'ci95_low':   round(np.mean(aucs)-1.96*np.std(aucs)/np.sqrt(n_splits),4),
        'ci95_high':  round(np.mean(aucs)+1.96*np.std(aucs)/np.sqrt(n_splits),4),
        'overall_auc':round(roc_auc_score(y, preds),4),
        'mean_f1':    round(np.mean(f1s),4),
        'mean_brier': round(np.mean(briers),4),
    }

# Load data
df_raw = pd.read_csv(DATA_PATH)
df_raw['text'] = df_raw['text'].fillna('').astype(str)
df_raw, _ = assign_roles(df_raw)
df_raw['is_bc'] = df_raw['text'].apply(
    lambda t: len(str(t).strip()) <= 3 or bool(BC_RE_STRICT.match(str(t))))

print(f'Loaded {len(df_raw):,} turns / {df_raw["conversation_id"].nunique()} conversations')
print(f'Substantive turns: {(~df_raw["is_bc"]).sum():,}')

---
## Section 1 — Data and Role Validation

In [ ]:
print('='*60)
print('SECTION 1: DATA AND ROLE VALIDATION')
print('='*60)

results_s1 = []

# 1a. Buyer speaks first
first_spk_is_buyer = sum(
    1 for cid, grp in df_raw.groupby('conversation_id')
    if grp.sort_values('start_time').iloc[0]['role'] == 'buyer')
results_s1.append({'check':'buyer_speaks_first','n':first_spk_is_buyer,
                    'pct':round(first_spk_is_buyer/178*100,1)})

# 1b. Seller anchors price first
seller_first = 0
for cid, grp in df_raw.groupby('conversation_id'):
    for _, row in grp.sort_values('start_time').iterrows():
        if PRICE_RE.search(str(row['text'])):
            if row['role'] == 'seller': seller_first += 1
            break
results_s1.append({'check':'seller_anchors_price_first','n':seller_first,
                    'pct':round(seller_first/178*100,1)})

# 1c. Outcome distribution
oc = df_raw.groupby('conversation_id')['outcome'].first().value_counts()
for oc_name, count in oc.items():
    results_s1.append({'check':f'outcome_{oc_name}','n':count,
                        'pct':round(count/178*100,1)})

# 1d. Very short conversations (<5 substantive turns)
sub_counts = df_raw[~df_raw['is_bc']].groupby('conversation_id').size()
short_convs = (sub_counts < 5).sum()
results_s1.append({'check':'very_short_convs_lt5_turns','n':int(short_convs),
                    'pct':round(short_convs/178*100,1)})

# 1e. Sale rate with vs without short convs
sale_all  = (df_raw.groupby('conversation_id')['outcome'].first()=='sale').mean()
valid_ids = sub_counts[sub_counts >= 5].index
sale_filt = (df_raw[df_raw['conversation_id'].isin(valid_ids)]
             .groupby('conversation_id')['outcome'].first()=='sale').mean()
results_s1.append({'check':'sale_rate_all',      'n':round(sale_all,3),  'pct':None})
results_s1.append({'check':'sale_rate_excl_short','n':round(sale_filt,3),'pct':None})

s1_df = pd.DataFrame(results_s1)
s1_df.to_csv(VAL_DIR + 'section1_data_validation.csv', index=False)
print(s1_df.to_string(index=False))

---
## Section 2 — Text Preprocessing Variants

In [ ]:
print('='*60)
print('SECTION 2: PREPROCESSING VARIANTS')
print('='*60)

BC_RE_LOOSE = re.compile(
    r'^\s*(yeah|mhm|mm|um|uh|okay|ok|yes|no|hi|hello|so|right|sure|huh|'
    r'ah|oh|hmm|yep|nope|alright|well|hey|got it|uh huh|i see|and|but)\s*$',
    re.IGNORECASE)
PRESERVED_RE = re.compile(
    r'\b(sounds good|okay deal|i can do that|deal|agreed)\b', re.IGNORECASE)

configs = [
    ('strict_bc',  lambda t: len(str(t).strip())<=3 or bool(BC_RE_STRICT.match(str(t)))),
    ('loose_bc',   lambda t: len(str(t).strip())<=3 or bool(BC_RE_LOOSE.match(str(t)))),
    ('len3_only',  lambda t: len(str(t).strip())<=3),
    ('no_filter',  lambda t: False),
]

results_s2 = []
for name, is_bc_fn in configs:
    sub = df_raw[~df_raw['text'].apply(is_bc_fn)].copy()
    preserved = sub['text'].apply(lambda t: bool(PRESERVED_RE.search(str(t)))).sum()
    # Quick sparsity check on approval
    approval_pat = re.compile(r'\b(agree|okay|sure|deal|right)\b', re.IGNORECASE)
    sparsity = 1 - sub['text'].apply(lambda t: bool(approval_pat.search(str(t)))).mean()
    results_s2.append({'config':name, 'n_turns':len(sub),
                        'pct_retained':round(len(sub)/len(df_raw)*100,1),
                        'preserved_negotiation_kw':int(preserved),
                        'approx_emotion_sparsity':round(sparsity*100,1)})
    print(f'  {name:<12}: {len(sub):,} turns  preserved_kw={preserved}  sparsity≈{round(sparsity*100,1)}%')

s2_df = pd.DataFrame(results_s2)
s2_df.to_csv(VAL_DIR + 'section2_preprocessing.csv', index=False)
print('\nRecommendation: strict_bc — negligible difference, cleaner signal.')

---
## Section 1B — GoEmotions Neural Encoder (GPU)

Replaces seed-lexicon with `SamLowe/roberta-base-go_emotions` neural classifier.
Outputs 28 continuous probability scores per turn (multi-label sigmoid).
Requires GPU — uses T4 in Colab free tier (~8 min for 7,565 turns).

In [ ]:
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device     = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'SamLowe/roberta-base-go_emotions'

print(f'Device: {device}')
if device == 'cpu':
    print('WARNING: No GPU detected. GoEmotions will run on CPU (~45 min).')
    print('Go to Runtime → Change runtime type → T4 GPU for best performance.')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(device).eval()

# Verify label order matches GOEMO_28
model_labels = [model.config.id2label[i] for i in range(len(model.config.id2label))]
assert model_labels == GOEMO_28, f'Label mismatch: {model_labels}'
print(f'Model labels verified: {len(model_labels)} dims')

In [ ]:
GOEMO_CACHE = OUT_DIR + 'stage1_goemotions_features.csv'

if os.path.exists(GOEMO_CACHE):
    print(f'Loading cached GoEmotions features from {GOEMO_CACHE}')
    goemo_df = pd.read_csv(GOEMO_CACHE)
else:
    print('Running GoEmotions inference...')

    def predict_goemo(texts, batch_size=64, max_length=128):
        all_probs = []
        for i in tqdm(range(0, len(texts), batch_size), desc='GoEmotions'):
            batch = texts[i:i+batch_size]
            enc   = tokenizer(batch, padding=True, truncation=True,
                               max_length=max_length, return_tensors='pt').to(device)
            with torch.no_grad():
                probs = torch.sigmoid(model(**enc).logits).cpu().numpy()
            all_probs.append(probs)
        return np.vstack(all_probs)

    texts     = df_raw['text'].tolist()
    goemo_probs = predict_goemo(texts, batch_size=64)

    goemo_cols = {f'goemo_{lbl}': goemo_probs[:, i]
                  for i, lbl in enumerate(GOEMO_28)}
    goemo_df   = pd.concat([
        df_raw[['conversation_id','speaker_id','start_time','end_time',
                'text','outcome','role','is_bc']].reset_index(drop=True),
        pd.DataFrame(goemo_cols)
    ], axis=1)
    goemo_df.to_csv(GOEMO_CACHE, index=False)
    print(f'Saved to {GOEMO_CACHE}')

GOEMO_COLS_23 = [f'goemo_{d}' for d in DIMS_23]
print(f'GoEmotions feature matrix: {goemo_df.shape}')
print(f'Mean activation (all turns): {goemo_df[GOEMO_COLS_23].values.mean():.4f}')
print(f'Sparsity (<0.05): {(goemo_df[GOEMO_COLS_23].values < 0.05).mean()*100:.1f}%')

In [ ]:
# ── GoEmotions quality checks ──────────────────────────────────────────────
sub_goemo = goemo_df[~goemo_df['is_bc']].copy()

print('GoEmotions activation stats (substantive turns):')
print(f'  Mean prob per dim: {sub_goemo[GOEMO_COLS_23].mean().mean():.4f}')
print(f'  Dims >0.1 per turn: {(sub_goemo[GOEMO_COLS_23] > 0.1).sum(axis=1).mean():.2f}')
print(f'  Sparsity (<0.05):   {(sub_goemo[GOEMO_COLS_23] < 0.05).mean()*100:.1f}%')

print('\nTop 5 buyer-seller emotion differences (GoEmotions):')
buyer_mean  = sub_goemo[sub_goemo['role']=='buyer'][GOEMO_COLS_23].mean()
seller_mean = sub_goemo[sub_goemo['role']=='seller'][GOEMO_COLS_23].mean()
diff = (buyer_mean - seller_mean).abs().sort_values(ascending=False)
for col in diff.head(5).index:
    em = col.replace('goemo_','')
    print(f'  {em:<18}: buyer={buyer_mean[col]:.4f}  '
          f'seller={seller_mean[col]:.4f}  Δ={diff[col]:.4f}')

print('\nDominant emotion distribution (all turns):')
dom = goemo_df[GOEMO_COLS_23].idxmax(axis=1).str.replace('goemo_','').value_counts()
print(dom.head(8).to_string())

---
## Section 3 — Emotion Measure Comparison

In [ ]:
print('='*60)
print('SECTION 3: EMOTION MEASURE COMPARISON')
print('='*60)

sub_raw = df_raw[~df_raw['is_bc']].copy().reset_index(drop=True)
conv_ids = sorted(sub_raw['conversation_id'].unique())
y_all    = np.array([1 if sub_raw[sub_raw['conversation_id']==cid]['outcome'].iloc[0]=='sale'
                     else 0 for cid in conv_ids])

def conv_mean_std(sub, cols):
    rows = []
    for cid in conv_ids:
        g = sub[sub['conversation_id']==cid]
        v = g[cols].values
        rows.append(np.concatenate([v.mean(axis=0), v.std(axis=0)+1e-8]))
    return np.array(rows)

results_s3 = []

# M1: Seed lexicon 23-dim
sub_lex = lexicon_features(sub_raw, window=3, dims=DIMS_23)
E_lex   = sub_lex[DIMS_23].values
ev_lex  = evaluate_classifier(conv_mean_std(sub_lex, DIMS_23), y_all)
results_s3.append({'measure':'seed_lexicon_23dim','n_dims':23,
                    'sparsity':(E_lex==0).mean()*100, **ev_lex})
print(f'  seed_lexicon   AUC={ev_lex["mean_auc"]:.4f} ± {ev_lex["std_auc"]:.4f}')

# M2: GoEmotions 23-dim (neural)
sub_goe = goemo_df[~goemo_df['is_bc']].copy().reset_index(drop=True)
# Apply rolling smoothing same as Stage 1A for fair comparison
sub_goe = sub_goe.sort_values(['conversation_id','start_time']).reset_index(drop=True)
for col in GOEMO_COLS_23:
    sub_goe[col] = sub_goe.groupby('conversation_id')[col].transform(
        lambda x: x.rolling(3, min_periods=1, center=True).mean())
E_goe   = sub_goe[GOEMO_COLS_23].values
ev_goe  = evaluate_classifier(conv_mean_std(sub_goe, GOEMO_COLS_23), y_all)
results_s3.append({'measure':'goemotions_neural_23dim','n_dims':23,
                    'sparsity':(E_goe<0.05).mean()*100, **ev_goe})
print(f'  goemotions     AUC={ev_goe["mean_auc"]:.4f} ± {ev_goe["std_auc"]:.4f}')

# M3: VADER proxy 3-dim
pos_re = re.compile(r'\b(good|great|excellent|perfect|happy|agree|deal|fair|yes)\b', re.IGNORECASE)
neg_re = re.compile(r'\b(no|not|bad|terrible|disagree|refuse|unfortunately|worried)\b', re.IGNORECASE)
sub_raw['vader_pos'] = sub_raw['text'].apply(lambda t: len(pos_re.findall(str(t))))
sub_raw['vader_neg'] = sub_raw['text'].apply(lambda t: len(neg_re.findall(str(t))))
sub_raw['vader_cmp'] = (sub_raw['vader_pos']-sub_raw['vader_neg']) / \
                        (sub_raw['vader_pos']+sub_raw['vader_neg']+1)
ev_vdr = evaluate_classifier(conv_mean_std(sub_raw, ['vader_pos','vader_neg','vader_cmp']), y_all)
results_s3.append({'measure':'vader_proxy_3dim','n_dims':3,'sparsity':None,**ev_vdr})
print(f'  vader_proxy    AUC={ev_vdr["mean_auc"]:.4f} ± {ev_vdr["std_auc"]:.4f}')

# M4: Negotiation-specific 8-dim
nego_sigs = {
    'firmness':   r'\b(firm|final offer|bottom line|will not|cannot|stand by)\b',
    'hesitation': r'\b(maybe|perhaps|not sure|I think|I guess|possibly)\b',
    'urgency':    r'\b(need to|must|have to|today|deadline)\b',
    'resistance': r'\b(no|not|never|refuse|reject|cannot accept)\b',
    'agreement':  r'\b(agree|deal|yes|okay|sure|sounds good|correct)\b',
    'concession': r'\b(willing to|could do|drop to|come down|go up|meet you)\b',
    'inquiry':    r'\b(what|how|tell me|could you|would you|why)\b',
    'rapport':    r'\b(great|nice|lovely|appreciate|thank you|pleased)\b',
}
for sig, pat in nego_sigs.items():
    sub_raw[sig] = sub_raw['text'].apply(
        lambda t: 1.0 if re.search(pat, str(t), re.IGNORECASE) else 0.0)
ev_nego = evaluate_classifier(conv_mean_std(sub_raw, list(nego_sigs.keys())), y_all)
results_s3.append({'measure':'nego_specific_8dim','n_dims':8,'sparsity':None,**ev_nego})
print(f'  nego_specific  AUC={ev_nego["mean_auc"]:.4f} ± {ev_nego["std_auc"]:.4f}')

s3_df = pd.DataFrame(results_s3)
s3_df.to_csv(VAL_DIR + 'section3_emotion_comparison.csv', index=False)

print('\n--- Section 3 summary ---')
print(s3_df[['measure','n_dims','mean_auc','std_auc','ci95_low','ci95_high']].to_string(index=False))

best = s3_df.loc[s3_df['mean_auc'].idxmax(), 'measure']
print(f'\nBest emotion measure: {best}')

---
## Section 4 — SSM Parameter Grid
> k = 2–6, smoothing window = 1, 3, 5 → 15 configurations

In [ ]:
def run_kalman_em(sequences, k, p=46, max_iter=60, tol=0.5):
    eps = 1e-6
    A   = np.eye(k)*0.9 + np.random.randn(k,k)*0.01
    C   = np.random.randn(p,k)*0.05
    Q   = np.eye(k)*0.1
    R   = np.eye(p)*0.3
    mu0 = np.zeros(k)
    V0  = np.eye(k)
    prev_ll = -np.inf

    for itr in range(max_iter):
        T_tot=0; s_zz=np.zeros((k,k)); s_zt_zt1=np.zeros((k,k))
        s_zt1_zt1=np.zeros((k,k)); s_xz=np.zeros((p,k))
        s_xx=np.zeros((p,p)); mu0_sum=np.zeros(k); V0_sum=np.zeros((k,k))
        total_ll=0.0; all_sm=[]

        for X in sequences:
            T=len(X)
            mu_f=np.zeros((T,k)); V_f=np.zeros((T,k,k))
            mu_p=np.zeros((T,k)); V_p=np.zeros((T,k,k))
            for t in range(T):
                mp = mu0 if t==0 else A@mu_f[t-1]
                Vp = V0  if t==0 else A@V_f[t-1]@A.T+Q
                mu_p[t]=mp; V_p[t]=Vp
                S  = C@Vp@C.T+R
                Si = np.linalg.pinv(S)
                K  = Vp@C.T@Si
                inn= X[t]-C@mp
                mu_f[t]=(np.eye(k)-K@C)@mp+K@X[t]
                V_f[t] =(np.eye(k)-K@C)@Vp
                sd,ld=np.linalg.slogdet(S)
                if sd>0: total_ll-=0.5*(ld+inn@Si@inn+p*np.log(2*np.pi))
            mu_s=mu_f.copy(); V_s=V_f.copy(); Vt_s=np.zeros((T-1,k,k))
            for t in range(T-2,-1,-1):
                G=V_f[t]@A.T@np.linalg.pinv(V_p[t+1])
                mu_s[t]=mu_f[t]+G@(mu_s[t+1]-mu_p[t+1])
                V_s[t] =V_f[t] +G@(V_s[t+1]-V_p[t+1])@G.T
                Vt_s[t]=V_s[t+1]@G.T
            all_sm.append(mu_s)
            T_tot+=T; mu0_sum+=mu_s[0]; V0_sum+=V_s[0]+np.outer(mu_s[0],mu_s[0])
            for t in range(T):
                ez=mu_s[t]; ezz=V_s[t]+np.outer(ez,ez)
                s_zz+=ezz; s_xz+=np.outer(X[t],ez); s_xx+=np.outer(X[t],X[t])
            for t in range(T-1):
                s_zt_zt1 +=Vt_s[t]+np.outer(mu_s[t+1],mu_s[t])
                s_zt1_zt1+=V_s[t] +np.outer(mu_s[t],mu_s[t])

        n=len(sequences)
        mu0=mu0_sum/n; V0=V0_sum/n-np.outer(mu0,mu0)+eps*np.eye(k)
        A  =s_zt_zt1@np.linalg.pinv(s_zt1_zt1+eps*np.eye(k))
        Q  =(s_zz-A@s_zt_zt1.T)/T_tot; Q=0.5*(Q+Q.T)+eps*np.eye(k)
        C  =s_xz@np.linalg.pinv(s_zz+eps*np.eye(k))
        Rf =((s_xx-C@s_xz.T)/T_tot); R=np.diag(np.maximum(np.diag(Rf),eps))*np.eye(p)
        delta=total_ll-prev_ll
        if itr>0 and abs(delta)<tol: break
        prev_ll=total_ll

    T_all=sum(len(X) for X in sequences)
    n_params=k*k+p*k+k*(k+1)//2+p
    return dict(A=A,C=C,Q=Q,R=R,mu0=mu0,V0=V0,ll=total_ll,
                bic=-2*total_ll+n_params*np.log(T_all),
                aic=-2*total_ll+2*n_params,
                sr=float(max(abs(np.linalg.eigvals(A)))),
                smoothed=all_sm,k=k)

print('SSM Kalman EM compiled.')

In [ ]:
print('='*60)
print('SECTION 4: SSM PARAMETER GRID (k=2..6, window=1,3,5)')
print('~15 configurations. Uses GoEmotions features if available.')
print('='*60)

# Use GoEmotions if available, else lexicon
USE_GOEMO = os.path.exists(GOEMO_CACHE)
print(f'Emotion source: {"GoEmotions (neural)" if USE_GOEMO else "Seed lexicon"}')

results_s4 = []

for window, k in product([1, 3, 5], range(2, 7)):
    print(f'  k={k}, window={window}...', end=' ', flush=True)

    # Build feature matrix for this window
    if USE_GOEMO:
        sub_g = goemo_df[~goemo_df['is_bc']].copy()
        sub_g = sub_g.sort_values(['conversation_id','start_time']).reset_index(drop=True)
        for col in GOEMO_COLS_23:
            sub_g[col] = sub_g.groupby('conversation_id')[col].transform(
                lambda x: x.rolling(window, min_periods=1, center=True).mean())
        em_col = GOEMO_COLS_23
        em_df  = sub_g
    else:
        sub_l = df_raw[~df_raw['is_bc']].copy()
        em_df = lexicon_features(sub_l, window=window, dims=DIMS_23)
        em_col = DIMS_23

    em_df['turn'] = em_df.groupby('conversation_id').cumcount()
    cids_4   = sorted(em_df['conversation_id'].unique())
    outcomes_4 = [1 if em_df[em_df['conversation_id']==cid]['outcome'].iloc[0]=='sale'
                  else 0 for cid in cids_4]

    sequences_4 = []
    for cid in cids_4:
        grp = em_df[em_df['conversation_id']==cid].sort_values('turn')
        T   = len(grp)
        bE  = np.zeros((T,23)); sE=np.zeros((T,23))
        lb  = np.zeros(23);    ls=np.zeros(23)
        for ti, (_, row) in enumerate(grp.iterrows()):
            ev = row[em_col].values.astype(float)
            if row['role']=='buyer': lb=ev
            else: ls=ev
            bE[ti]=lb; sE[ti]=ls
        sequences_4.append(np.hstack([bE,sE]))

    try:
        res = run_kalman_em(sequences_4, k=k, p=46, max_iter=50, tol=1.0)
        feats_4 = []
        for z_seq in res['smoothed']:
            T = len(z_seq)
            feats_4.append(np.concatenate([
                z_seq.mean(axis=0), z_seq.std(axis=0)+1e-8,
                z_seq[max(0,2*T//3):].mean(axis=0)]))
        ev_4 = evaluate_classifier(np.array(feats_4), np.array(outcomes_4))
        results_s4.append({'window':window,'k':k,
                            'bic':round(res['bic'],1),'aic':round(res['aic'],1),
                            'log_lik':round(res['ll'],1),'spectral_r':round(res['sr'],4),
                            **{f'{k2}':v for k2,v in ev_4.items()}})
        print(f"BIC={res['bic']:.0f}  sr={res['sr']:.3f}  AUC={ev_4['mean_auc']:.4f}")
    except Exception as e:
        print(f'FAILED: {e}')
        results_s4.append({'window':window,'k':k,'error':str(e)})

s4_df = pd.DataFrame(results_s4)
s4_df.to_csv(VAL_DIR + 'section4_ssm_grid.csv', index=False)

best4 = s4_df.dropna(subset=['bic']).loc[s4_df.dropna(subset=['bic'])['bic'].idxmin()]
print(f"\nBest by BIC: k={best4['k']}, window={best4['window']}, "
      f"BIC={best4['bic']}, AUC={best4.get('mean_auc','n/a')}")

---
## Section 5 — Bargaining Act Annotation Validation

In [ ]:
print('='*60)
print('SECTION 5: ANNOTATION VALIDATION')
print('='*60)

ANN_PATH = OUT_DIR + 'stage3_annotated.csv'

if not os.path.exists(ANN_PATH):
    print('stage3_annotated.csv not found.')
    print('Run stage3/stage3_annotation.py first, then re-run this cell.')
else:
    ann = pd.read_csv(ANN_PATH)
    sub_ann = ann[~ann['is_bc']].copy()
    total   = len(sub_ann)

    results_s5 = []
    for col, name in [('act_push','push'),('act_comparison','comparison'),
                       ('act_new_offer','new_offer'),('act_allowance','allowance'),
                       ('act_end','end')]:
        r_sale   = sub_ann[sub_ann['outcome']=='sale'][col].mean()
        r_nosale = sub_ann[sub_ann['outcome']!='sale'][col].mean()
        results_s5.append({'act':name,
                            'n':int(sub_ann[col].sum()),
                            'pct':round(sub_ann[col].mean()*100,1),
                            'rate_sale':round(r_sale*100,1),
                            'rate_nosale':round(r_nosale*100,1),
                            'delta_pp':round((r_sale-r_nosale)*100,2)})

    for sub_t in ['push_constraint','push_disparagement','push_neutral']:
        r_s = (sub_ann[sub_ann['outcome']=='sale']['push_subtype']==sub_t).mean()
        r_n = (sub_ann[sub_ann['outcome']!='sale']['push_subtype']==sub_t).mean()
        n   = (sub_ann['push_subtype']==sub_t).sum()
        results_s5.append({'act':sub_t,'n':int(n),'pct':round(n/total*100,2),
                            'rate_sale':round(r_s*100,2),'rate_nosale':round(r_n*100,2),
                            'delta_pp':round((r_s-r_n)*100,2)})

    # End act position check
    end_turns = sub_ann[sub_ann['act_end']==1].copy()
    end_turns['t_idx']   = end_turns.groupby('conversation_id').cumcount()
    end_turns['conv_len']= end_turns['conversation_id'].map(
        sub_ann.groupby('conversation_id').size())
    true_end = (end_turns['t_idx']/end_turns['conv_len'] >= 0.8).mean()
    results_s5.append({'act':'end_in_last_20pct',
                        'n':len(end_turns),'pct':round(true_end*100,1),
                        'rate_sale':None,'rate_nosale':None,'delta_pp':None})

    s5_df = pd.DataFrame(results_s5)
    s5_df.to_csv(VAL_DIR + 'section5_annotation_validation.csv', index=False)
    print(s5_df.to_string(index=False))

---
## Section 6 — Outcome Prediction Model Grid

In [ ]:
print('='*60)
print('SECTION 6: OUTCOME PREDICTION MODEL GRID')
print('='*60)

sub_raw2 = df_raw[~df_raw['is_bc']].copy().reset_index(drop=True)
cids_6   = sorted(sub_raw2['conversation_id'].unique())
y_6      = np.array([1 if sub_raw2[sub_raw2['conversation_id']==cid]['outcome'].iloc[0]=='sale'
                     else 0 for cid in cids_6])

def conv_feat(sub, cols):
    rows = []
    for cid in cids_6:
        g = sub[sub['conversation_id']==cid]
        v = np.nan_to_num(g[cols].values.astype(float))
        T = len(v)
        rows.append(np.concatenate([v.mean(axis=0), v.std(axis=0)+1e-8,
                                     v[max(0,2*T//3):].mean(axis=0)]))
    return np.array(rows)

# Price features
price_rows = []
for cid in cids_6:
    grp = sub_raw2[sub_raw2['conversation_id']==cid]
    ps  = [p for t in grp['text'] for p in extract_prices(t)]
    price_rows.append([ps[-1] if ps else 230, ps[0] if ps else 230,
                        abs(ps[-1]-ps[0]) if len(ps)>1 else 0, len(ps),
                        grp['end_time'].max()-grp['start_time'].min()])
F_price = np.array(price_rows)

# Duration baseline
dur_rows = []
for cid in cids_6:
    grp = sub_raw2[sub_raw2['conversation_id']==cid]
    dur_rows.append([grp['end_time'].max()-grp['start_time'].min(),
                      len(grp), (grp['role']=='buyer').mean()])
F_dur = np.array(dur_rows)

# Lexicon features
sub_lex6 = lexicon_features(sub_raw2, window=3, dims=DIMS_23)
F_lex    = conv_feat(sub_lex6, DIMS_23)

# GoEmotions features
sub_goe6 = goemo_df[~goemo_df['is_bc']].copy().reset_index(drop=True)
sub_goe6 = sub_goe6.sort_values(['conversation_id','start_time']).reset_index(drop=True)
for col in GOEMO_COLS_23:
    sub_goe6[col] = sub_goe6.groupby('conversation_id')[col].transform(
        lambda x: x.rolling(3, min_periods=1, center=True).mean())
F_goe = conv_feat(sub_goe6, GOEMO_COLS_23)

# Load SSM + tactic features from disk if available
LAT_PATH  = OUT_DIR + 'stage2_latent_states.csv'
TACT_PATH = OUT_DIR + 'stage3_annotated.csv'
has_ssm   = os.path.exists(LAT_PATH) and os.path.exists(TACT_PATH)

models_6 = {
    'majority_class':   F_dur,
    'duration_turns':   F_dur,
    'price_only':       F_price,
    'lexicon_23dim':    F_lex,
    'goemotions_23dim': F_goe,
    'price_lexicon':    np.hstack([F_price, F_lex]),
    'price_goemo':      np.hstack([F_price, F_goe]),
}

if has_ssm:
    lat  = pd.read_csv(LAT_PATH)
    ann6 = pd.read_csv(TACT_PATH)
    ann6 = ann6[~ann6['is_bc']].copy()
    ann6['turn'] = ann6.groupby('conversation_id').cumcount()
    merged6 = ann6.merge(lat, on=['conversation_id','turn'], how='inner')
    z_cols  = ['z_1','z_2','z_3','z_4']
    ssm_rows, tact_rows = [], []
    for cid in cids_6:
        g  = merged6[merged6['conversation_id']==cid]
        ga = ann6[ann6['conversation_id']==cid]
        T  = len(g)
        Z  = g[z_cols].values if T > 0 else np.zeros((1,4))
        ssm_rows.append(np.concatenate([Z.mean(axis=0), Z.std(axis=0)+1e-8,
                                         Z[max(0,2*T//3):].mean(axis=0)]))
        Ta = len(ga)
        tact_rows.append([
            ga['act_push'].sum()/Ta, ga['act_comparison'].sum()/Ta,
            ga['act_allowance'].sum()/Ta, ga['act_new_offer'].sum()/Ta,
            (ga['push_subtype']=='push_constraint').sum()/Ta,
            (ga['push_subtype']=='push_disparagement').sum()/Ta,
            (ga['comp_subtype']=='comparison_mixed').sum()/Ta,
        ])
    F_ssm  = np.array(ssm_rows)
    F_tact = np.array(tact_rows)
    models_6['ssm_latent']          = F_ssm
    models_6['price_ssm']           = np.hstack([F_price, F_ssm])
    models_6['goemo_ssm']           = np.hstack([F_goe, F_ssm])
    models_6['price_ssm_tactics']   = np.hstack([F_price, F_ssm, F_tact])
    models_6['goemo_ssm_tactics']   = np.hstack([F_goe, F_ssm, F_tact])

results_s6 = []
for mname, F in models_6.items():
    ev = evaluate_classifier(F, y_6)
    ev['model'] = mname
    ev['n_feats'] = F.shape[1]
    results_s6.append(ev)
    print(f'  {mname:<30}: AUC={ev["mean_auc"]:.4f}±{ev["std_auc"]:.4f}  '
          f'CI=[{ev["ci95_low"]:.4f},{ev["ci95_high"]:.4f}]  F1={ev["mean_f1"]:.4f}')

s6_df = pd.DataFrame(results_s6)
s6_df.to_csv(VAL_DIR + 'section6_outcome_prediction.csv', index=False)

---
## Section 7 — Robustness Checks

In [ ]:
print('='*60)
print('SECTION 7: ROBUSTNESS CHECKS')
print('='*60)

conv_meta = sub_raw2.groupby('conversation_id').agg(
    n_turns=('text','count'), duration=('end_time','max'),
    outcome=('outcome','first'), buyer_pct=('role', lambda x: (x=='buyer').mean())
).reset_index()
conv_meta['sale'] = (conv_meta['outcome']=='sale').astype(int)
conv_meta['dur_tertile'] = pd.qcut(conv_meta['duration'], 3, labels=['short','medium','long'])

# Use GoEmotions features for robustness checks
sub_goe7 = sub_goe6.copy()
results_s7 = []

# 7a. By duration tertile
for tertile in ['short','medium','long']:
    ids   = conv_meta[conv_meta['dur_tertile']==tertile]['conversation_id'].values
    sub_t = sub_goe7[sub_goe7['conversation_id'].isin(ids)]
    y_t   = conv_meta[conv_meta['dur_tertile']==tertile]['sale'].values
    if len(y_t) < 10 or y_t.sum() < 3: continue
    rows  = []
    for cid in ids:
        g = sub_t[sub_t['conversation_id']==cid]
        rows.append(g[GOEMO_COLS_23].mean().values)
    ev = evaluate_classifier(np.array(rows), y_t, n_splits=min(5,int(y_t.sum())))
    results_s7.append({'check':f'dur_{tertile}','n':len(ids),
                        'sale_rate':round(y_t.mean()*100,1), **ev})
    print(f'  dur={tertile:<6}: n={len(ids)}  AUC={ev["mean_auc"]:.4f}')

# 7b. Emotion + duration control
ctrl_rows = []
for cid in cids_6:
    row = conv_meta[conv_meta['conversation_id']==cid].iloc[0]
    g   = sub_goe7[sub_goe7['conversation_id']==cid]
    ctrl_rows.append(np.concatenate([
        g[GOEMO_COLS_23].mean().values,
        [float(row['duration']), float(row['n_turns']), float(row['buyer_pct'])]
    ]))
ev_ctrl = evaluate_classifier(np.array(ctrl_rows), y_6)
results_s7.append({'check':'goemo_plus_duration','n':len(cids_6),
                    'sale_rate':round(y_6.mean()*100,1), **ev_ctrl})
print(f'  goemo+duration control: AUC={ev_ctrl["mean_auc"]:.4f}')

# 7c. Exclude outlier conversations (>2 std duration)
dm, ds = conv_meta['duration'].mean(), conv_meta['duration'].std()
no_out = conv_meta[(conv_meta['duration']>=dm-2*ds) &
                    (conv_meta['duration']<=dm+2*ds)]['conversation_id'].values
y_no   = conv_meta[conv_meta['conversation_id'].isin(no_out)]['sale'].values
rows_no = [sub_goe7[sub_goe7['conversation_id']==cid][GOEMO_COLS_23].mean().values
            for cid in no_out]
ev_no  = evaluate_classifier(np.array(rows_no), y_no)
results_s7.append({'check':'excl_outlier_duration','n':int(len(no_out)),
                    'sale_rate':round(y_no.mean()*100,1), **ev_no})
print(f'  excl outlier duration: n={len(no_out)}  AUC={ev_no["mean_auc"]:.4f}')

# 7d. Approval-token check: are results driven by okay/sure/agree?
APPROVAL_RE = re.compile(r'^\s*(okay|sure|right|yes|agree|deal)\s*$', re.IGNORECASE)
sub_noa = sub_goe7[~sub_goe7['text'].apply(lambda t: bool(APPROVAL_RE.match(str(t))))]
dims_noa = [c for c in GOEMO_COLS_23 if 'approval' not in c]
rows_noa = []
for cid in cids_6:
    g = sub_noa[sub_noa['conversation_id']==cid]
    rows_noa.append(g[dims_noa].mean().values if len(g)>0 else np.zeros(len(dims_noa)))
ev_noa = evaluate_classifier(np.array(rows_noa), y_6)
results_s7.append({'check':'excl_approval_tokens','n':len(cids_6),
                    'sale_rate':round(y_6.mean()*100,1), **ev_noa})
print(f'  excl approval tokens:  AUC={ev_noa["mean_auc"]:.4f}')

s7_df = pd.DataFrame(results_s7)
s7_df.to_csv(VAL_DIR + 'section7_robustness.csv', index=False)

---
## Section 8–9 — Decision Rule: RETAIN / EXPLORATORY / DISCARD

In [ ]:
print('='*60)
print('SECTION 8–9: DECISION TABLE')
print('Criteria (study-validation-list.txt §9):')
print('  RETAIN      : CI_low > 0.50  AND  std_auc < 0.10')
print('  EXPLORATORY : CI_low > 0.50  AND  std_auc >= 0.10')
print('  DISCARD     : CI_low <= 0.50')
print('='*60)

price_auc = s6_df[s6_df['model']=='price_only']['mean_auc'].values
price_auc = price_auc[0] if len(price_auc) > 0 else 0.539

decisions = []
for _, row in s6_df.iterrows():
    auc      = row.get('mean_auc', 0)
    ci_low   = row.get('ci95_low', 0)
    std_auc  = row.get('std_auc', 1)
    above_05 = ci_low > 0.50
    stable   = std_auc < 0.10

    if above_05 and stable:   verdict = 'RETAIN'
    elif above_05:            verdict = 'EXPLORATORY'
    else:                     verdict = 'DISCARD'

    note = ''
    if auc < 0.60:
        note = '⚠ AUC < 0.60: treat as exploratory, avoid causal language'
    if auc > price_auc:
        note += ' ✓ beats price baseline'

    decisions.append({'model':row.get('model',''), 'mean_auc':round(auc,4),
                       'ci95_low':round(ci_low,4), 'std_auc':round(std_auc,4),
                       'beats_price': auc > price_auc,
                       'stable': stable, 'verdict':verdict, 'note':note})

s89_df = pd.DataFrame(decisions)
s89_df.to_csv(VAL_DIR + 'section89_decision_table.csv', index=False)

print(f"\n{'Model':<30} {'AUC':>6} {'CI_low':>7} {'Std':>6} {'Stable':>7} {'Verdict':>12}")
print('-'*72)
for _, r in s89_df.sort_values('mean_auc', ascending=False).iterrows():
    print(f"{r['model']:<30} {r['mean_auc']:>6.4f} {r['ci95_low']:>7.4f} "
          f"{r['std_auc']:>6.4f} {'Yes' if r['stable'] else 'No':>7} {r['verdict']:>12}")

retain  = s89_df[s89_df['verdict']=='RETAIN']['model'].tolist()
explore = s89_df[s89_df['verdict']=='EXPLORATORY']['model'].tolist()
discard = s89_df[s89_df['verdict']=='DISCARD']['model'].tolist()
print(f'\nRETAIN      : {retain}')
print(f'EXPLORATORY : {explore}')
print(f'DISCARD     : {discard}')

---
## Save All Results and Push to GitHub

In [ ]:
# Summary of all output files
print('Validation outputs:')
for f in sorted(os.listdir(VAL_DIR)):
    path = os.path.join(VAL_DIR, f)
    size = os.path.getsize(path)
    print(f'  {f:<45} {size:>8,} bytes')

print('\nAll sections complete.')

In [ ]:
# ── Push results to GitHub ─────────────────────────────────────────────────
from google.colab import userdata

GITHUB_TOKEN = userdata.get('GITHUB_API_KEY')

%cd /content/optimal-nego

!git config --global user.email "shashank.dubey@hec.edu"
!git config --global user.name "catchshashank"
!git config pull.rebase true

REMOTE = f'https://{GITHUB_TOKEN}@github.com/catchshashank/optimal-nego.git'
!git remote set-url origin {REMOTE}

!git pull origin main
!git add outputs/validation/ outputs/stage1_goemotions_features.csv
!git commit -m "Add validation pipeline results and GoEmotions features"
!git push origin main

print('Pushed to github.com/catchshashank/optimal-nego')